# PICKO Research · NB3 — **Separation**: can it tell look-alike tools apart?

We finetune one model on the 40 focus tools and probe curated groups of near-identical tools (same action
across sources, or same source across actions). Each group's queries are drawn from the model's **held-out
test set** and offered the full group in one shot, so we measure pure disambiguation + which tool it
confuses for which.

## 0 · Colab quick-start (GPU) — run & forget, restart-safe

**On Colab first: Runtime → Change runtime type → GPU (L4 recommended; T4/A100 also fine).**
This cell clones **upstream Needle** (Cactus, pinned commit) and installs it, clones this **PICKO** repo,
pins the exact JAX/Flax, mounts Drive, and points **both** the data (in) and the checkpoints+results (out)
at your **`MyDrive/picko/`** folder — so a runtime restart loses nothing.

**Prerequisite (one-time):** `picko_training_pool.jsonl` must be in `MyDrive/picko/`. **Running locally?**
This cell is a no-op — install Needle yourself (`pip install -e /path/to/needle`) and skip to cell 1.

In [ ]:
# --- Colab bootstrap (safe to re-run; no-op locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
# Upstream Needle (Cactus) — de-vendored: cloned + installed at a pinned commit.
NEEDLE_REPO = "https://github.com/cactus-compute/needle.git"
NEEDLE_SHA  = "34861f39ae292429f80a62c96abe83218a852d57"   # pinned; has _per_tool_split — update if upstream drifts
# This PICKO repo — the scripts/notebooks/data imported below.
PICKO_REPO   = "https://github.com/hodayastern/Picko.git"  # this submission repo
PICKO_BRANCH = "main"
if IN_COLAB:
    if not os.path.exists("/content/needle"):
        !git clone -q {NEEDLE_REPO} /content/needle && cd /content/needle && git checkout -q {NEEDLE_SHA}
    if not os.path.exists("/content/picko"):
        !git clone -q -b {PICKO_BRANCH} {PICKO_REPO} /content/picko
    %pip install -q "jax[cuda12]==0.10.2" "jaxlib==0.10.2" "flax==0.12.8"
    %pip install -q -e /content/needle                          # install upstream Needle
    sys.path.insert(0, "/content/picko")
    from google.colab import drive; drive.mount("/content/drive")
    import shutil
    DRIVE = "/content/drive/MyDrive/picko"                      # <- everything lives here
    os.environ["PICKO_OUT_DIR"] = f"{DRIVE}/picko_out"          # checkpoints + results (durable)
    os.environ["PICKO_LOG"]     = f"{DRIVE}/picko_out/run.log"  # durable log across restarts
    os.makedirs(os.environ["PICKO_OUT_DIR"], exist_ok=True)
    dst = "/content/picko/data/picko_training_pool.jsonl"
    if not os.path.exists(dst):
        cands = [f"{DRIVE}/picko_training_pool.jsonl", "/content/drive/MyDrive/picko_training_pool.jsonl"]
        src = next((c for c in cands if os.path.exists(c)), None)
        if src is None:
            have = os.listdir(DRIVE) if os.path.isdir(DRIVE) else "(MyDrive/picko not found)"
            raise FileNotFoundError(
                "picko_training_pool.jsonl not found. Upload it to MyDrive/picko/. "
                f"Currently in {DRIVE}: {have}")
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst)
        print("copied data from", src)
    try:                                                        # guard: upstream must expose the split PICKO uses
        from needle.training.finetune import _per_tool_split    # noqa: F401
    except Exception as e:
        raise ImportError(f"Upstream Needle @ {NEEDLE_SHA[:7]} lacks _per_tool_split ({e}). "
                          "Pin NEEDLE_SHA to a commit that has it, or re-run the install.")
    import jax
    print("GPU:");
    !nvidia-smi -L
    print("jax devices:", jax.devices())
    _plat = jax.devices()[0].platform
    assert _plat == "gpu", (
        f"JAX is running on '{_plat}', NOT the GPU — every finetune/eval will be ~30x slower "
        "(hours instead of minutes). FIX: Runtime > Change runtime type > GPU (L4), then "
        "Runtime > Restart session, and re-run this cell. If a GPU IS selected but this still "
        "fails, the CUDA plugin didn't load — re-run the %pip lines above, then restart.")
    print(f"bootstrap OK · GPU active · needle@{NEEDLE_SHA[:7]} · data =", dst, "· OUT_DIR =", os.environ["PICKO_OUT_DIR"])
else:
    print("Not on Colab — running locally. Install upstream Needle first: pip install -e /path/to/needle")

## 1 · Setup & data overview

In [ ]:
# ensure the repo root is importable (works from notebooks/research/, Colab, etc.)
import os, sys
_here = os.path.abspath(os.getcwd())
for _ in range(6):
    if os.path.exists(os.path.join(_here, "scripts", "picko_research.py")): break
    _here = os.path.dirname(_here)
if os.path.isdir("/content/picko"): _here = "/content/picko"
if _here not in sys.path: sys.path.insert(0, _here)

from scripts.picko_research import *
import json, time
import pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid")
except Exception:
    sns = None
from tqdm.auto import tqdm

cat, tok, raw, FOCUS, OUT_DIR = load_context()
env_report(OUT_DIR)   # jax devices + is OUT_DIR durable (Drive)?

### The 40 focus tools\nOne row per tool, with its family, category and **parameter count / bucket**.

In [ ]:
display(tools_dataframe(cat, FOCUS))

### All examples for these 40 tools\nOne row per training example (query → gold tool), tagged with the gold tool's **param bucket**.

In [ ]:
ex_df = examples_dataframe(cat, raw, FOCUS)
print("examples:", ex_df.shape[0], "| per param bucket:", ex_df["param_bucket"].value_counts().to_dict())
display(ex_df.head(10))

## 2 · Train / test split

Both the training subprocess and this notebook call the **same deterministic** `per_tool_split`
(`seed=42`, 10 test + 10 val per tool). The model trains **only on the train split**; the group probes
below run **only on the held-out test split**, so no test query is ever seen in training.

## 3 · The ambiguous groups

Each group is a set of look-alike tools on one of two axes: **cross-source** (same action, different
source — the source word disambiguates) or **within-source** (same source, subtly different action).

In [ ]:
AXIS = {"cross_source_search": "cross-source", "single_item_summary": "cross-source",
        "hf_search_variants": "within-source", "wikipedia_retrieve_vs_summarize": "within-source",
        "get_paper_content": "within-source", "arxiv_latex": "within-source"}
grp_rows = []
for g, tools in SIMILAR_GROUPS.items():
    fams = sorted({family_of(t) for t in tools})
    grp_rows.append({"group": g, "axis": AXIS.get(g, ""), "n_tools": len(tools),
                     "families": ",".join(fams), "tools": ", ".join(tools)})
groups_df = pd.DataFrame(grp_rows).sort_values("axis")
display(groups_df)

### Sample queries per tool\nThe actual requests the model must tell apart — one example query per tool, grouped.

In [ ]:
def gold_of(ex):
    calls = json.loads(ex.get("answers", "[]"))
    return next((c["name"] for c in calls if isinstance(c, dict) and c.get("name")), None)

sample_q = {}
for e in raw:
    g = gold_of(e)
    if g and g not in sample_q: sample_q[g] = e["query"]

ex_rows = [{"group": g, "axis": AXIS.get(g, ""), "tool": t, "example_query": sample_q.get(t, "")[:160]}
           for g, tools in SIMILAR_GROUPS.items() for t in tools]
display(pd.DataFrame(ex_rows))

## 4 · Train the model (40 focus tools)\nTrained once to Drive and reused; the returned `test` set is the held-out split the group probes run on.

In [ ]:
NB_DIR = os.path.join(OUT_DIR, "nb3_separation"); os.makedirs(NB_DIR, exist_ok=True)   # this notebook's outputs
CAP_PER_TOOL, EPOCHS, BATCH_SIZE = 120, 1, 8   # examples/tool -> 100 train / 10 val / 10 test; BATCH_SIZE: lower to 4 on OOM
RUN_TRAIN, FORCE_RETRAIN = True, False
FOCUS40 = finetune_and_eval(cat, raw, tok, FOCUS, "focus40", NB_DIR,
                            cap=CAP_PER_TOOL, epochs=EPOCHS, compact=False, token_aware=True,
                            eval_subsample=None, run_train=RUN_TRAIN,
                            force_retrain=FORCE_RETRAIN, batch_size=BATCH_SIZE)
m40, p40, tk40 = FOCUS40["bundle"]
TEST = FOCUS40["test"]                     # held-out test queries (never trained on)
log(f"focus40 selection_acc={FOCUS40['metrics']['selection_acc']:.3f} · held-out test={len(TEST)}")

## 5 · Per-group disambiguation\nFor each group we take the held-out queries whose gold tool is in the group and offer the full group. Resumable: finished groups persist to `nb3_separation/separation_results.json`.

In [ ]:
import contextlib, io
RES = os.path.join(NB_DIR, "separation_results.json")
prev = json.load(open(RES)) if (os.path.exists(RES) and not FORCE_RETRAIN) else {"per_group": [], "confusion": {}}
sep_by = {r["group"]: r for r in prev.get("per_group", [])}
group_conf = prev.get("confusion", {})
if sep_by: log(f"resumed {len(sep_by)} finished group(s)")

def gold_of(ex):
    calls = json.loads(ex.get("answers", "[]"))
    return next((c["name"] for c in calls if isinstance(c, dict) and c.get("name")), None)

t_all = time.time()
for gname, gtools in SIMILAR_GROUPS.items():
    if gname in sep_by and gname in group_conf and not FORCE_RETRAIN:
        log(f"{gname}: skip (done)"); continue
    try:
        gset = set(gtools)
        gtest = [e for e in TEST if gold_of(e) in gset]                      # held-out queries for this group
        offered = [offer_subset(cat, gold_of(e), gtools, len(gtools), seed=0, compact=False) for e in gtest]
        with contextlib.redirect_stdout(io.StringIO()):
            gpreds = predict(m40, p40, tk40, gtest, tools_override=offered)
        gm = evaluate(gtest, gpreds, family_of=family_of)
        sep_by[gname] = {"group": gname, "n_tools": len(gtools), "n": gm["n"],
                         "selection_acc": gm["selection_acc"], "name_f1": gm["name_f1"]}
        group_conf[gname] = confusion(gtest, gpreds)
        json.dump({"per_group": list(sep_by.values()), "confusion": group_conf}, open(RES, "w"), indent=2)
        log(f"{gname}: selection={gm['selection_acc']:.3f} (n={gm['n']})")
    except Exception as e:
        log(f"{gname}: FAILED ({type(e).__name__}: {e})")

log(f"ALL GROUPS DONE in {time.time()-t_all:.0f}s · results={RES}")
if not sep_by:
    raise RuntimeError("No group succeeded — see the FAILED lines above.")
separation = pd.DataFrame(list(sep_by.values())).sort_values("selection_acc")
display(separation)

plt.figure(figsize=(8,4)); plt.barh(separation["group"], separation["selection_acc"], color="#4C72B0")
plt.xlim(0,1); plt.xlabel("tool-selection accuracy"); plt.title("Separation: hardest look-alike groups (lower = more confused)")
plt.tight_layout(); save_fig("separation_groups", out_dir=NB_DIR); plt.show()

## 6 · Confusion heatmaps (who gets mistaken for whom)

In [ ]:
for gname, conf in group_conf.items():
    labels = sorted(set(conf) | {p for row in conf.values() for p in row})
    M = pd.DataFrame(0, index=sorted(conf), columns=labels)
    for r, row in conf.items():
        for p, n in row.items(): M.loc[r, p] = n
    plt.figure(figsize=(0.9*len(labels)+2, 0.5*len(M)+1.5))
    if sns: sns.heatmap(M, annot=True, fmt="d", cmap="Blues", cbar=False)
    else:
        plt.imshow(M.values, cmap="Blues"); plt.xticks(range(len(labels)), labels, rotation=90); plt.yticks(range(len(M)), M.index)
    plt.title(f"Separation · {gname}"); plt.xlabel("predicted"); plt.ylabel("reference")
    plt.tight_layout(); save_fig(f"separation_confusion_{gname}", out_dir=NB_DIR); plt.show()

## 7 · Read-out

Residual selection errors concentrate inside these look-alike groups. The lowest-accuracy group is the
frontier for a tool-picker; the heatmaps show whether confusions are symmetric (two tools mutually
confused) or a sink (everything collapses to one generic tool).